In [93]:
# -----------------------------
# 1단계. 기본 집합 및 참여자 데이터 구성
# -----------------------------

# SKU 집합
P = ["P1", "P2", "P3", "P4"]

# SKU 정보
# 특정 도메인에 고정하지 않고 item1, item2 형태로 일반화
skus = {
    "P1": {"name": "item1"},
    "P2": {"name": "item2"},
    "P3": {"name": "item3"},
    "P4": {"name": "item4"},
}

In [94]:
# -----------------------------
# 1-1. 화주 데이터
# -----------------------------

# inventory: 현재 가용 재고
# ask_price: 화주가 최소한 받고 싶은 SKU 단위 가격
# beta: 화주 쪽 최소 거래비율

sellers = {
    "S1": {
        "name": "seller1",
        "sku_data": {
            "P1": {
                "inventory": 300,
                "ask_price": 4000,
                "beta": 0.30
            },
            "P2": {
                "inventory": 200,
                "ask_price": 4200,
                "beta": 0.20
            }
        }
    },

    "S2": {
        "name": "seller2",
        "sku_data": {
            "P2": {
                "inventory": 250,
                "ask_price": 4300,
                "beta": 0.30
            },
            "P3": {
                "inventory": 250,
                "ask_price": 4400,
                "beta": 0.30
            }
        }
    },

    "S3": {
        "name": "seller3",
        "sku_data": {
            "P3": {
                "inventory": 150,
                "ask_price": 4500,
                "beta": 0.40
            },
            "P4": {
                "inventory": 500,
                "ask_price": 4100,
                "beta": 0.10
            }
        }
    }
}

In [95]:
# -----------------------------
# 1-2. 구매자 주문 데이터
# -----------------------------

# budget: 주문 전체 예산 상한
# priority: 주문 처리 우선순위
# split_allowed: 여러 화주로부터 분할 이행 허용 여부
# demand: SKU별 요청 수량
# unit_bid: 구매자가 해당 SKU 1개에 대해 지불할 수 있는 최대 가격
# alpha: SKU별 최소 충족률

buyers = {
    "B1": {
        "name": "buyer1",
        "orders": {
            "O1": {
                "budget": 4000000,
                "priority": 0.90,
                "split_allowed": True,
                "items": {
                    "P1": {
                        "demand": 300,
                        "unit_bid": 6000,
                        "alpha": 1.0
                    },
                    "P2": {
                        "demand": 200,
                        "unit_bid": 5800,
                        "alpha": 0.7
                    },
                    "P4": {
                        "demand": 300,
                        "unit_bid": 5200,
                        "alpha": 0.0
                    }
                }
            }
        }
    },

    "B2": {
        "name": "buyer2",
        "orders": {
            "O2": {
                "budget": 3000000,
                "priority": 0.75,
                "split_allowed": False,
                "items": {
                    "P2": {
                        "demand": 200,
                        "unit_bid": 5900,
                        "alpha": 1.0
                    },
                    "P3": {
                        "demand": 150,
                        "unit_bid": 6100,
                        "alpha": 0.8
                    }
                }
            }
        }
    },

    "B3": {
        "name": "buyer3",
        "orders": {
            "O3": {
                "budget": 5000000,
                "priority": 0.60,
                "split_allowed": True,
                "items": {
                    "P1": {
                        "demand": 100,
                        "unit_bid": 5700,
                        "alpha": 1.0
                    },
                    "P3": {
                        "demand": 200,
                        "unit_bid": 6000,
                        "alpha": 0.6
                    },
                    "P4": {
                        "demand": 200,
                        "unit_bid": 5300,
                        "alpha": 0.0
                    }
                }
            }
        }
    }
}

In [96]:
# -----------------------------
# 1-3. 운송 및 fulfillment 데이터
# -----------------------------

# capacity: 현재 처리 가능 수량
# unit_cost: SKU 1개당 운송 및 fulfillment 비용

transporters = {
    "K1": {
        "name": "carrier1",
        "capacity": 700,
        "unit_cost": 500
    },

    "K2": {
        "name": "carrier2",
        "capacity": 500,
        "unit_cost": 650
    }
}

In [97]:
# -----------------------------
# 2단계. 현재 처리할 주문 선택
# -----------------------------

def select_next_order(buyers):
    """
    priority가 가장 높은 주문을 선택한다.
    순차 처리 구조이므로 한 번에 하나의 주문만 처리한다.
    """

    order_pool = []

    for b, bdata in buyers.items():
        for o, odata in bdata["orders"].items():
            order_pool.append((b, o, odata))

    selected_b, selected_o, selected_order = max(
        order_pool,
        key=lambda x: x[2]["priority"]
    )

    return selected_b, selected_o, selected_order


b, o, order = select_next_order(buyers)

print("Selected buyer:", b)
print("Selected order:", o)

Selected buyer: B1
Selected order: O1


In [98]:
# -----------------------------
# 3단계. M 및 거래 가능 tuple 생성
# -----------------------------

def build_valid_tuples(order, sellers, transporters):
    """
    현재 주문에 대해 거래 가능한 (seller, sku, carrier) tuple만 생성한다.

    거래 가능 조건:
    buyer unit_bid >= seller ask_price + carrier unit_cost

    M[(s,p,k)]:
    해당 조합에서 이행 가능한 최대 수량
    """

    valid_tuples = []
    M = {}

    for p, item in order["items"].items():

        demand = item["demand"]
        buyer_bid = item["unit_bid"]

        for s, sdata in sellers.items():

            # 화주 s가 SKU p를 보유하지 않으면 제외
            if p not in sdata["sku_data"]:
                continue

            inventory = sdata["sku_data"][p]["inventory"]
            seller_ask = sdata["sku_data"][p]["ask_price"]

            for k, kdata in transporters.items():

                capacity = kdata["capacity"]
                carrier_cost = kdata["unit_cost"]

                # 거래 가능성 조건
                if buyer_bid >= seller_ask + carrier_cost:

                    m_value = min(demand, inventory, capacity)

                    if m_value > 0:
                        valid_tuples.append((s, p, k))
                        M[(s, p, k)] = m_value

    return valid_tuples, M


valid_tuples, M = build_valid_tuples(order, sellers, transporters)

print("Valid tuples")
for tup in valid_tuples:
    print(tup, "M =", M[tup])

Valid tuples
('S1', 'P1', 'K1') M = 300
('S1', 'P1', 'K2') M = 300
('S1', 'P2', 'K1') M = 200
('S1', 'P2', 'K2') M = 200
('S2', 'P2', 'K1') M = 200
('S2', 'P2', 'K2') M = 200
('S3', 'P4', 'K1') M = 300
('S3', 'P4', 'K2') M = 300


In [99]:
# -----------------------------
# 4단계. Gurobi 모델 생성
# -----------------------------

import gurobipy as gp
from gurobipy import GRB

model = gp.Model("sequential_many_to_one_matching")

# 현재 주문에 포함된 SKU 집합
P_o = list(order["items"].keys())

# 유효한 화주-SKU pair
valid_seller_sku_pairs = list(
    set((s, p) for (s, p, k) in valid_tuples)
)

# 유효한 운송사 집합
valid_carriers = list(
    set(k for (s, p, k) in valid_tuples)
)

In [100]:
# -----------------------------
# 5단계. 의사결정변수
# -----------------------------

# x: 현재 주문 수락 여부
x = model.addVar(
    vtype=GRB.BINARY,
    name="x_order"
)

# q[s,p,k]: 화주 s의 SKU p를 운송사 k를 통해 이행하는 수량
# 거래 가능한 tuple에 대해서만 생성
q = model.addVars(
    valid_tuples,
    vtype=GRB.CONTINUOUS,
    lb=0,
    name="q"
)

# z[s,p,k]: 해당 화주-SKU-운송사 조합 사용 여부
z = model.addVars(
    valid_tuples,
    vtype=GRB.BINARY,
    name="z"
)

# y[s,p]: 화주 s의 SKU p lot 사용 여부
y = model.addVars(
    valid_seller_sku_pairs,
    vtype=GRB.BINARY,
    name="y"
)

# u[k]: 운송사 k 사용 여부
u = model.addVars(
    valid_carriers,
    vtype=GRB.BINARY,
    name="u"
)

# Q[p]: SKU p의 총 이행 수량
Q = model.addVars(
    P_o,
    vtype=GRB.CONTINUOUS,
    lb=0,
    name="Q"
)

# h[p]: SKU p의 미충족 수량
h = model.addVars(
    P_o,
    vtype=GRB.CONTINUOUS,
    lb=0,
    name="h"
)

In [101]:
# -----------------------------
# 6단계. 목적식
# -----------------------------

# 부족 penalty 설정
# alpha = 0인 선택 품목은 미충족 penalty를 0으로 둔다.
lambda_penalty = {}

for p, item in order["items"].items():
    if item["alpha"] == 0:
        lambda_penalty[p] = 0
    else:
        lambda_penalty[p] = 0.5 * item["unit_bid"]


# 플랫폼 이익
welfare_expr = gp.quicksum(
    (
        order["items"][p]["unit_bid"]
        - sellers[s]["sku_data"][p]["ask_price"]
        - transporters[k]["unit_cost"]
    ) * q[s, p, k]
    for (s, p, k) in valid_tuples
)

# 부분 수락 또는 미충족 penalty
shortage_penalty_expr = gp.quicksum(
    lambda_penalty[p] * h[p]
    for p in P_o
)

model.setObjective(
    welfare_expr - shortage_penalty_expr,
    GRB.MAXIMIZE
)

In [102]:
# -----------------------------
# 7-1. SKU별 총 이행 수량 정의
# -----------------------------

for p in P_o:
    model.addConstr(
        Q[p] == gp.quicksum(
            q[s, p2, k]
            for (s, p2, k) in valid_tuples
            if p2 == p
        ),
        name=f"total_fulfilled_{p}"
    )

In [103]:
# -----------------------------
# 7-2. 최소 충족률 제약
# -----------------------------

for p, item in order["items"].items():
    model.addConstr(
        Q[p] >= item["alpha"] * item["demand"] * x,
        name=f"min_fulfillment_{p}"
    )

In [104]:
# -----------------------------
# 7-3. 수요 초과 이행 방지
# -----------------------------

for p, item in order["items"].items():
    model.addConstr(
        Q[p] <= item["demand"] * x,
        name=f"max_demand_{p}"
    )

In [105]:
# -----------------------------
# 7-4. 부족 수량 정의
# -----------------------------

for p, item in order["items"].items():
    model.addConstr(
        h[p] == item["demand"] * x - Q[p],
        name=f"shortage_{p}"
    )

In [106]:
# -----------------------------
# 7-5. 화주 재고 제약
# -----------------------------

for (s, p) in valid_seller_sku_pairs:
    model.addConstr(
        gp.quicksum(
            q[s2, p2, k]
            for (s2, p2, k) in valid_tuples
            if s2 == s and p2 == p
        )
        <= sellers[s]["sku_data"][p]["inventory"],
        name=f"inventory_{s}_{p}"
    )

In [107]:
# -----------------------------
# 7-6. 운송 capacity 제약
# -----------------------------

for k in valid_carriers:
    model.addConstr(
        gp.quicksum(
            q[s, p, k2]
            for (s, p, k2) in valid_tuples
            if k2 == k
        )
        <= transporters[k]["capacity"] * u[k],
        name=f"capacity_{k}"
    )

In [108]:
# -----------------------------
# 7-7. q-z 연결 제약
# -----------------------------

for (s, p, k) in valid_tuples:
    model.addConstr(
        q[s, p, k] <= M[(s, p, k)] * z[s, p, k],
        name=f"link_q_z_{s}_{p}_{k}"
    )

In [109]:
# -----------------------------
# 7-8. z-y 연결 제약
# -----------------------------

for (s, p, k) in valid_tuples:
    model.addConstr(
        z[s, p, k] <= y[s, p],
        name=f"link_z_y_{s}_{p}_{k}"
    )

In [110]:
# -----------------------------
# 7-9. 화주 최소 거래비율 제약
# -----------------------------

for (s, p) in valid_seller_sku_pairs:

    beta = sellers[s]["sku_data"][p]["beta"]
    inventory = sellers[s]["sku_data"][p]["inventory"]

    model.addConstr(
        gp.quicksum(
            q[s2, p2, k]
            for (s2, p2, k) in valid_tuples
            if s2 == s and p2 == p
        )
        >= beta * inventory * y[s, p],
        name=f"min_trade_ratio_{s}_{p}"
    )

In [111]:
# -----------------------------
# 7-10. z-u 연결 제약
# -----------------------------

for (s, p, k) in valid_tuples:
    model.addConstr(
        z[s, p, k] <= u[k],
        name=f"link_z_u_{s}_{p}_{k}"
    )

In [112]:
# -----------------------------
# 7-11. 구매자 예산 제약
# -----------------------------

model.addConstr(
    gp.quicksum(
        order["items"][p]["unit_bid"] * Q[p]
        for p in P_o
    )
    <= order["budget"],
    name="buyer_budget"
)

<gurobi.Constr *Awaiting Model Update*>

In [113]:
# -----------------------------
# 8단계. 최적화 실행
# -----------------------------

model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5900X 12-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 47 rows, 29 columns and 112 nonzeros (Max)
Model fingerprint: 0xa60fd131
Model has 10 linear objective coefficients
Variable types: 14 continuous, 15 integer (15 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+03]
  Objective range  [5e+02, 3e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+02, 4e+06]

Found heuristic solution: objective -0.0000000
Presolve removed 31 rows and 16 columns
Presolve time: 0.00s
Presolved: 16 rows, 13 columns, 49 nonzeros
Variable types: 9 continuous, 4 integer (4 binary)

Root relaxation: objective 7.900000e+05, 10 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | In

In [114]:
# -----------------------------
# 9단계. 결과 출력
# -----------------------------

if model.status == GRB.OPTIMAL:

    print("Social welfare value: ", model.objVal)
    print("Order accepted:", round(x.X))

    print("\nSKU fulfillment result")
    for p in P_o:
        print(
            p,
            skus[p]["name"],
            "| demand:", order["items"][p]["demand"],
            "| alpha:", order["items"][p]["alpha"],
            "| fulfilled:", Q[p].X,
            "| shortage:", h[p].X
        )

    print("\nAssignment result")
    for (s, p, k) in valid_tuples:
        if q[s, p, k].X > 1e-6:
            unit_surplus = (
                order["items"][p]["unit_bid"]
                - sellers[s]["sku_data"][p]["ask_price"]
                - transporters[k]["unit_cost"]
            )

            print(
                f"{s}-{p}-{k}",
                "| item:", skus[p]["name"],
                "| quantity:", q[s, p, k].X,
                "| unit_surplus:", unit_surplus
            )

else:
    print("No optimal solution found.")
    print("Status:", model.status)

Social welfare value:  790000.0
Order accepted: 1

SKU fulfillment result
P1 item1 | demand: 300 | alpha: 1.0 | fulfilled: 300.0 | shortage: 0.0
P2 item2 | demand: 200 | alpha: 0.7 | fulfilled: 200.0 | shortage: 0.0
P4 item4 | demand: 300 | alpha: 0.0 | fulfilled: 200.0 | shortage: 100.0

Assignment result
S1-P1-K1 | item: item1 | quantity: 300.0 | unit_surplus: 1500
S1-P2-K1 | item: item2 | quantity: 200.0 | unit_surplus: 1100
S3-P4-K1 | item: item4 | quantity: 200.0 | unit_surplus: 600


In [115]:
# -----------------------------
# 10단계. 재고 및 capacity 업데이트
# -----------------------------

def update_state_after_matching(sellers, transporters, q, valid_tuples):
    """
    현재 주문 처리 결과를 반영하여
    화주 재고와 운송 capacity를 업데이트한다.
    """

    for (s, p, k) in valid_tuples:

        used_qty = q[s, p, k].X

        if used_qty > 1e-6:
            sellers[s]["sku_data"][p]["inventory"] -= used_qty
            transporters[k]["capacity"] -= used_qty

    return sellers, transporters


if model.status == GRB.OPTIMAL and x.X > 0.5:
    sellers, transporters = update_state_after_matching(
        sellers,
        transporters,
        q,
        valid_tuples
    )